In [27]:
# uncomment following line to make an html
!jupyter nbconvert --to html gothaam_ana_summary_plots.ipynb

[NbConvertApp] Converting notebook gothaam_ana_summary_plots.ipynb to html
[NbConvertApp] Writing 2860765 bytes to gothaam_ana_summary_plots.html


# Table of Contents

1. Profiles by time of day

In [3]:
import importlib
import gothaam_utils
import math
import numpy as np
import netCDF4
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, ListedColormap
from metpy.plots import SkewT
from metpy.units import pandas_dataframe_to_unit_arrays, units
from datetime import datetime, timedelta
from IPython.display import display
from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row
from bokeh.layouts import gridplot
from bokeh.plotting import figure, show
from bokeh.models import Title, CustomJS, Select, TextInput, Button, LinearAxis, Range1d
from bokeh.models.formatters import DatetimeTickFormatter
from bokeh.models.tickers import DatetimeTicker
from bokeh.palettes import Category10
import warnings
import itertools 
import holoviews as hv 
from holoviews import dim, opts
import hvplot.pandas
import copy
hv.extension('bokeh', 'matplotlib')
warnings.filterwarnings('ignore')
output_notebook()
%matplotlib inline

Loading BokehJS ...

In [4]:
# Open all netcdf files and read them into data frames
importlib.reload(gothaam_utils)

data_dir = '/Users/ckruse/qaqc/GOTHAAM/data/v9'
project = 'GOTHAAM'

# open all NetCDF files
nc_dict = gothaam_utils.open_nc(data_dir) # Dictionary of flight NetCDFs

# read the variables selected in gothaam_utils.py from each NetCDF file
data_1hz = {} # dictionary of DataFrame's
for flight, nc in nc_dict.items():
    data_1hz[flight] = gothaam_utils.read_nc(nc)
    print(f"Done reading {flight}")

Found 0 proficiency flights, 2 ferry flights, 2 test flights, and 21 research flights
Opening all flight NetCDF Files
This file contains PRELIMINARY DATA that are NOT to be used for critical analysis.
NIDAS version: v1.2.5-74
NetCDF: Attribute not found
Processing Date & Time: 2026-02-26T00:51:50 +0000
Done reading ff01
Done reading ff02
Done reading rf01
Done reading rf02
Done reading rf03
Done reading rf04
Done reading rf05
Done reading rf06
Done reading rf07
Done reading rf08
Done reading rf09
Done reading rf10
Done reading rf11
Done reading rf12
Done reading rf13
Done reading rf14
Done reading rf15
Done reading rf16
Done reading rf17
Done reading rf18
Done reading rf19
Done reading rf20
Done reading rf21
Done reading tf01
Done reading tf02


# Detecting Climbs and Descents

In [5]:
importlib.reload(gothaam_utils)
all_profiles = {}
for flight in data_1hz.keys():
    if not "rf" in flight:
        continue
    profiles = gothaam_utils.detect_climb_descent(data_1hz[flight])
    land_profiles = [pair for pair in profiles if gothaam_utils.is_land_profile(data_1hz[flight],pair[0],pair[1])]
    marine_profiles = [pair for pair in profiles if not gothaam_utils.is_land_profile(data_1hz[flight],pair[0],pair[1])]
    all_profiles[flight] = {'land_profiles': land_profiles, 'marine_profiles': marine_profiles}

## How Many Profiles Were Performed?

In [8]:
n_land_profs_all = 0
n_marine_profs_all = 0
for flight in data_1hz.keys():
    if not "rf" in flight:
        continue
    n_land_profs = len(all_profiles[flight]['land_profiles'])
    n_marine_profs = len(all_profiles[flight]['marine_profiles'])
    n_tot_profs = n_land_profs +  n_marine_profs
    n_land_profs_all = n_land_profs_all + n_land_profs
    n_marine_profs_all = n_marine_profs_all + n_marine_profs
    #print(f"Flight: {flight}, # Land Profiles: {n_land_profs:02d}, # Marine Profiles: {n_marine_profs:02d}, Total: {n_tot_profs:02d}")

n_tot_profs_all = n_land_profs_all + n_marine_profs_all
print(f"Totals: # Land Profiles: {n_land_profs_all:02d}, # Marine Profiles: {n_marine_profs_all:02d}, Total: {n_tot_profs_all:02d}")


Totals: # Land Profiles: 318, # Marine Profiles: 265, Total: 583


In [24]:
# Reload the utilities
importlib.reload(gothaam_utils)

# Plot Land Profiles
gothaam_utils.plot_profiles_by_time_split(
    data_1hz, 
    all_profiles, 
    domain='land', 
    title='Land THETA Profiles'
)

# Plot Marine Profiles
gothaam_utils.plot_profiles_by_time_split(
    data_1hz, 
    all_profiles, 
    domain='marine', 
    title='Marine THETA Profiles'
)